## **Stacking and Blending Ensembles**

### **Topic Roadmap**

**1. Prepare a binary classification dataset**

**2. Fit a stacking classifier**

**3. Implement a holdout blending model**

**4. Compare ensemble performance**

**5. Key revision notes**

## **1. Dataset and Split**

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=RANDOM_STATE
)

## **2. Stacking Classifier**

Stacking trains a meta-model on out-of-fold predictions from base estimators. The final estimator learns how to combine their signals.

In [2]:
estimators = [
    ("rf", RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))),
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5, stack_method="predict_proba", n_jobs=-1
)
stack.fit(X_train, y_train)
print(f"Stacking accuracy: {stack.score(X_test, y_test):.3f}")

Stacking accuracy: 0.965


## **3. Holdout Blending**

Blending uses a separate validation holdout. Base predictions on that holdout train a meta-model, which is then evaluated on the untouched test split.

In [3]:
X_base, X_blend, y_base, y_blend = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)
base_models = [
    RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
]
for model in base_models:
    model.fit(X_base, y_base)
blend_features = np.column_stack([model.predict_proba(X_blend)[:, 1] for model in base_models])
test_features = np.column_stack([model.predict_proba(X_test)[:, 1] for model in base_models])

In [4]:
meta_model = LogisticRegression(max_iter=2000).fit(blend_features, y_blend)
blend_predictions = (meta_model.predict_proba(test_features)[:, 1] >= 0.5).astype(int)
print(f"Blending accuracy: {accuracy_score(y_test, blend_predictions):.3f}")

Blending accuracy: 0.947


### **Key Revision Notes**

- Stacking uses cross-validated base predictions by default.
- Blending uses a holdout set to train the meta-model.
- The meta-model must not train on predictions produced from the same rows used to fit the base model.
- More models do not guarantee improvement; diversity and validation design matter.